In [1]:
import os, re, sqlite3
import pandas as pd

DATA_DIR = '/content/data'   # adjust if your folder lives elsewhere

mdb = sqlite3.connect(':memory:')

def mq(sql):
    '''Run a query against the Movies DB and return a DataFrame.'''
    return pd.read_sql_query(sql, mdb)

# Postgres -> SQLite tweaks applied as we read each .sql file
for f in sorted(os.listdir(DATA_DIR)):
    if not f.endswith('.sql'):
        continue
    with open(os.path.join(DATA_DIR, f)) as fh:
        sql = fh.read()
    sql = re.sub(r'DROP SCHEMA.*?CASCADE;', '', sql, flags=re.IGNORECASE | re.DOTALL)
    sql = re.sub(r'CREATE SCHEMA\s+\w+\s*;', '', sql, flags=re.IGNORECASE)
    sql = sql.replace('movies.', '')
    sql = re.sub(r'INT NOT NULL GENERATED BY DEFAULT AS IDENTITY', 'INTEGER', sql)
    sql = re.sub(r'varchar\(\d+\)', 'TEXT', sql, flags=re.IGNORECASE)
    sql = re.sub(r'decimal\(\d+,\d+\)', 'REAL', sql, flags=re.IGNORECASE)
    sql = re.sub(r'BIGINT', 'INTEGER', sql)
    sql = re.sub(r'date\s+DEFAULT', 'TEXT DEFAULT', sql)
    try:
        mdb.executescript(sql)
    except Exception:
        # Some files emit harmless 'no transaction' warnings after auto-commit; ignore.
        pass

# Sanity check
print('Movies DB loaded. Row counts:')
for t in ['movie', 'genre', 'person', 'production_company', 'keyword',
          'movie_genres', 'movie_cast', 'movie_crew', 'movie_company', 'movie_keywords']:
    try:
        n = mdb.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t:22s}: {n:>8,} rows')
    except Exception:
        print(f'  {t:22s}: MISSING')

Movies DB loaded. Row counts:
  movie                 :    4,803 rows
  genre                 :       20 rows
  person                :  104,842 rows
  production_company    :    5,047 rows
  keyword               :    9,794 rows
  movie_genres          :   12,160 rows
  movie_cast            :  106,257 rows
  movie_crew            :  129,581 rows
  movie_company         :   13,677 rows
  movie_keywords        :   36,162 rows


Exercise 1 : Movie Rankings and Analysis

Task 1 : Rank Movies by Popularity within Each Genre

In [2]:
mq('''
SELECT g.genre_name,
       m.title,
       m.popularity,
       RANK() OVER (PARTITION BY g.genre_id ORDER BY m.popularity DESC) AS rank_in_genre
FROM movie m
JOIN movie_genres mg ON mg.movie_id = m.movie_id
JOIN genre       g  ON  g.genre_id = mg.genre_id
ORDER BY g.genre_name, rank_in_genre
''')

,genre_name,title,popularity,rank_in_genre
0,Action,Deadpool,514.569956,1
1,Action,Guardians of the Galaxy,481.098624,2
2,Action,Mad Max: Fury Road,434.278564,3
3,Action,Jurassic World,418.708552,4
4,Action,Pirates of the Caribbean: The Curse of the Bla...,271.972889,5
...,...,...,...,...
12155,Western,The Ballad of Gregorio Cortez,0.592821,78
12156,Western,Western Religion,0.589540,79
12157,Western,Doc Holliday's Revenge,0.459400,80
12158,Western,All Hat,0.137535,81


Task 2 : Identify the Top 3 Movies by Revenue within Each Production Company

In [3]:
mq('''
SELECT pc.company_name,
        m.title,
        m.revenue,
        NTILE(4) OVER (PARTITION BY pc.company_id ORDER BY m.revenue DESC) AS revenue_quartile
FROM movie               m
JOIN movie_company       mc ON mc.movie_id = m.movie_id
JOIN production_company  pc ON pc.company_id = mc.company_id
WHERE m.revenue > 0
ORDER BY pc.company_name, revenue_quartile
''')

,company_name,title,revenue,revenue_quartile
0,1.85 Films,Rubber,98017,1
1,100 Bares,El secreto de sus ojos,33965843,1
2,100 Bares,Metegol,24000000,2
3,1019 Entertainment,Captive,2801508,1
4,10th Hole Productions,The Kids Are All Right,34705850,1
...,...,...,...,...
10731,uFilm,Astérix aux Jeux Olympiques,132900000,1
10732,uFilm,Zwartboek,26193068,2
10733,uFilm,Escobar: Paradise Lost,3758328,3
10734,uFilm,The Adventurer: The Curse of the Midas Box,6399,4


Task 3 : Calculate the Running Total of Movie Budgets for Each Genre

In [4]:

sql_running_total = """
SELECT
    g.genre_name,
    m.title,
    m.budget,
    SUM(m.budget) OVER (
        PARTITION BY g.genre_name
        ORDER BY m.release_date, m.title
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_budget
FROM genre g
JOIN movie_genres mg ON g.genre_id = mg.genre_id
JOIN movie m ON mg.movie_id = m.movie_id
WHERE m.budget > 0
ORDER BY g.genre_name, m.release_date;
"""

mq(sql_running_total)



,genre_name,title,budget,running_total_budget
0,Action,Hell's Angels,3950000,3950000
1,Action,The Charge of the Light Brigade,1200000,5150000
2,Action,Sands of Iwo Jima,1000000,6150000
3,Action,Annie Get Your Gun,3768785,9918785
4,Action,The Greatest Show on Earth,4000000,13918785
...,...,...,...,...
9865,Western,Forsaken,11000000,1956453601
9866,Western,The Ridiculous 6,60000000,2016453601
9867,Western,The Hateful Eight,44000000,2060453601
9868,Western,The Revenant,135000000,2195453601


Task 4 : Identify the Most Recent Movie for Each Genre

In [5]:
sql_most_recent = """
SELECT DISTINCT
    g.genre_name,
    FIRST_VALUE(m.title) OVER (
        PARTITION BY g.genre_name
        ORDER BY m.release_date DESC, m.title ASC
    ) AS most_recent_movie,
    FIRST_VALUE(m.release_date) OVER (
        PARTITION BY g.genre_name
        ORDER BY m.release_date DESC, m.title ASC
    ) AS release_date
FROM genre g
JOIN movie_genres mg ON g.genre_id = mg.genre_id
JOIN movie m ON mg.movie_id = m.movie_id
WHERE m.release_date IS NOT NULL AND m.release_date != ''
ORDER BY g.genre_name;
"""

mq(sql_most_recent)


,genre_name,most_recent_movie,release_date
0,Action,Suicide Squad,2016-08-02
1,Adventure,Kicks,2016-09-09
2,Animation,Sausage Party,2016-07-11
3,Comedy,Growing Up Smith,2017-02-03
4,Crime,Suicide Squad,2016-08-02
5,Documentary,"To Be Frank, Sinatra at 100",2015-12-12
6,Drama,Growing Up Smith,2017-02-03
7,Family,Growing Up Smith,2017-02-03
8,Fantasy,Pete's Dragon,2016-08-10
9,Foreign,Burn,2012-11-01


Exercise 2 : Cast and Crew Performance Analysis

Task 1: Rank Actors by Their Appearance in Movies



In [6]:
sql_task1 = """
SELECT
    p.person_name,
    COUNT(mc.movie_id) AS movie_count,
    DENSE_RANK() OVER (ORDER BY COUNT(mc.movie_id) DESC) AS actor_rank
FROM person p
JOIN movie_cast mc ON p.person_id = mc.person_id
GROUP BY p.person_id, p.person_name
ORDER BY actor_rank ASC
LIMIT 20;
"""
mq(sql_task1)


,person_name,movie_count,actor_rank
0,Samuel L. Jackson,67,1
1,Robert De Niro,57,2
2,Bruce Willis,51,3
3,Matt Damon,48,4
4,Morgan Freeman,46,5
5,Steve Buscemi,43,6
6,Liam Neeson,41,7
7,Johnny Depp,40,8
8,Owen Wilson,40,8
9,John Goodman,39,9


Task 2: Identify the Top Director by Average Movie Rating



In [7]:
sql_task2 = """
WITH DirectorRatings AS (
    SELECT
        p.person_name,
        AVG(m.vote_average) AS avg_rating
    FROM person p
    JOIN movie_crew mc ON p.person_id = mc.person_id
    JOIN movie m ON mc.movie_id = m.movie_id
    WHERE mc.job = 'Director' AND m.vote_count > 10 -- On filtre pour avoir des notes significatives
    GROUP BY p.person_id, p.person_name
)
SELECT person_name, avg_rating
FROM (
    SELECT
        person_name,
        avg_rating,
        RANK() OVER (ORDER BY avg_rating DESC) as rnk
    FROM DirectorRatings
)
WHERE rnk = 1;
"""
mq(sql_task2)


,person_name,avg_rating
0,John Cromwell,8.4
1,W.S. Van Dyke,8.4


Task 3: Calculate the Cumulative Revenue of Movies Acted by Each Actor



In [8]:
sql_task3 = """
SELECT
    p.person_name,
    m.title,
    m.revenue,
    SUM(m.revenue) OVER (
        PARTITION BY p.person_id
        ORDER BY m.release_date ASC
    ) AS cumulative_revenue
FROM person p
JOIN movie_cast mc ON p.person_id = mc.person_id
JOIN movie m ON mc.movie_id = m.movie_id
WHERE m.revenue > 0
ORDER BY p.person_name, m.release_date
LIMIT 20;
"""
mq(sql_task3)


,person_name,title,revenue,cumulative_revenue
0,Larry Mullen Jr.,U2 3D,22730842,22730842
1,'Snub' Pollard,Singin' in the Rain,7200000,7200000
2,'Snub' Pollard,Pocketful of Miracles,5000000,12200000
3,'Snub' Pollard,The Man Who Shot Liberty Valance,8000000,20200000
4,'Wild Bill' Laczko,Day of the Dead,34000000,34000000
5,50 Cent,Get Rich or Die Tryin',46442528,46442528
6,50 Cent,Righteous Kill,73174566,119617094
7,50 Cent,Morning Glory,58785180,178402274
8,50 Cent,The Frozen Ground,5496951,183899225
9,50 Cent,Escape Plan,122915111,306814336


Task 4: Identify the Director Whose Movies Have the Highest Total Budget



In [9]:
sql_task4 = """
WITH DirectorBudgets AS (
    SELECT
        p.person_name,
        SUM(m.budget) AS total_budget
    FROM person p
    JOIN movie_crew mc ON p.person_id = mc.person_id
    JOIN movie m ON mc.movie_id = m.movie_id
    WHERE mc.job = 'Director'
    GROUP BY p.person_id, p.person_name
)
SELECT person_name, total_budget
FROM (
    SELECT
        person_name,
        total_budget,
        RANK() OVER (ORDER BY total_budget DESC) as rnk
    FROM DirectorBudgets
)
WHERE rnk = 1;
"""
mq(sql_task4)


,person_name,total_budget
0,Steven Spielberg,1667500000
